[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/060.%20The%20Warehouse%20Slotting%20Optimization%20Problem/P60-Tier-5_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 60. The Warehouse Slotting Optimization Problem

## Tier 5: The Integrated Digital Twin

### Purpose and learning objectives

This notebook builds a small, synthetic warehouse digital twin to connect slotting decisions with operational simulation, demand forecasting, bottleneck analysis, and scenario comparison. A digital twin is a software representation of an operational system that is updated or informed by data about that system. Here, the warehouse, its SKUs, locations, pickers, transactions, and sensor readings are simulated; no live warehouse systems or sensor feeds are connected.

By following the notebook, you will learn how to:

1. Represent warehouse entities and operational data in Python.
2. Simulate order generation and picker activity over discrete time steps.
3. Summarize operational indicators (KPIs) and inspect their time trends.
4. Forecast SKU demand using a simple exponential-smoothing-style procedure.
5. Identify high-activity locations and locations with high forecast-based load.
6. Compare three slotting heuristics in a virtual scenario.

### Inputs, outputs, and scope

The main inputs are synthetic SKU demand rates, space requirements and weights; location capacities, weight limits and accessibility scores; and picker speeds. The code produces simulated transactions, KPI records, demand forecasts, bottleneck analyses, candidate slotting assignments, and visualizations.

The model is illustrative rather than a calibrated physical or operational digital twin. Accessibility is used as a simplified proxy for travel distance; travel is not routed through a warehouse map. Demand generation, picker behavior, sensors, forecasts, and performance estimates are synthetic. The model does not include order batching, inventory replenishment, congestion, SKU-location compatibility rules, labor shifts, or live WMS/IoT integration. A location is considered feasible based on space and weight capacity only. These assumptions are important when interpreting the results: outputs are demonstrations of the modeled procedures, not evidence of real-world performance or forecast accuracy.

### Slotting model and decision logic

Let $I$ denote the set of SKUs and $J$ the set of locations. For SKU $i$, let $v_i$ be its demand velocity, $s_i$ its space requirement, and $w_i$ its weight. For location $j$, let $C_j$ be its space capacity, $W_j$ its weight limit, and $a_j$ its accessibility score (smaller values indicate easier access). The assignment decision is represented by $x_{ij}$, which equals 1 when SKU $i$ is assigned to location $j$, and 0 otherwise.

The basic feasibility conditions are:

$$
\sum_{j \in J} x_{ij} \leq 1 \quad (i \in I), \qquad
\sum_{i \in I} s_i x_{ij} \leq C_j \quad (j \in J), \qquad
\sum_{i \in I} w_i x_{ij} \leq W_j \quad (j \in J), \qquad
x_{ij} \in \{0,1\}.
$$

The inequality for each SKU allows it to remain unassigned if no location has enough remaining capacity. The code uses greedy heuristics rather than a mathematical-programming solver: it sorts SKUs and locations, then assigns each SKU to the first location that satisfies both capacity checks. For the efficiency-oriented heuristic, a useful conceptual measure is the demand-weighted accessibility cost

$$
D(x) = \sum_{i \in I}\sum_{j \in J} v_i a_j x_{ij}.
$$

This expression explains why high-velocity SKUs are given priority for more accessible locations. The scenario performance calculation multiplies accessibility by 10 to form a simplified distance proxy, so the corresponding total is $10D(x)$ when all SKUs are assigned. More generally, the code sums this contribution over the SKUs actually assigned. The heuristic does not guarantee a globally optimal assignment. If $n=|I|$ and $m=|J|$, sorting takes $O(n\log n + m\log m)$ time and the greedy feasibility search takes at most $O(nm)$ time.

The three scenario heuristics retain distinct priorities: efficiency sorts by velocity and considers accessible locations first; space utilization assigns each SKU to a feasible location with the lowest current percentage utilization, with a randomized SKU order; balanced slotting sorts by velocity per unit space and scores feasible locations using accessibility and a utilization penalty. These procedures are scenario rules, not exact solutions to a common optimization objective.

### Notebook sequence

The notebook first defines the data structures and digital-twin simulation, then initializes and simulates a baseline. It next inspects performance and forecasts demand, defines the three slotting scenarios, adds common performance calculations, and finally compares the scenarios. Read scenario improvements as model-based estimates under the assumptions above. They are not measured operational savings.

### Interpreting the case figures

The accompanying problem description gives an illustrative MegaFulfill case with 50,000 daily transactions, 500 tested scenarios, an 18% picking-time reduction, 95% space utilization, 200 SKU relocations, and a 25% peak-period picking-time reduction. These figures are presented as case-description claims; they are not inputs to this notebook and are not generated or validated by its simulation. The notebook's synthetic example uses 200 SKUs, 30 locations, and 10 pickers.

### When this approach may be useful

A digital-twin-style workflow can support what-if analysis when an operation has reliable data, a sufficiently representative system model, and a way to validate simulated predictions against observed outcomes. Building and maintaining such a system can require substantial data integration, sensor infrastructure, computational resources, and operational expertise. A simulation alone does not establish predictive accuracy or guarantee that a proposed slotting plan will improve actual performance.

In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import random
from typing import Any, Dict, List, Optional
import warnings

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Fixed seeds make the synthetic example reproducible when the cells are run in order.
np.random.seed(42)
random.seed(42)

In [ ]:
@dataclass
class DigitalSKU:
    """SKU attributes and the demand history used by the forecast procedure."""

    id: int
    velocity: float
    space_req: float
    weight: float
    category: str
    price: float
    location_id: Optional[int] = None
    demand_history: List[float] = field(default_factory=list)
    last_updated: datetime = field(default_factory=datetime.now)
    sensor_readings: Dict[str, float] = field(default_factory=dict)


@dataclass
class DigitalLocation:
    """Location capacity, accessibility, assigned SKUs, and synthetic sensor data."""

    id: int
    capacity: float
    weight_limit: float
    accessibility: float
    zone: str
    current_space_used: float = 0.0
    current_weight_used: float = 0.0
    assigned_skus: List[int] = field(default_factory=list)
    sensor_data: Dict[str, Any] = field(default_factory=dict)
    utilization_history: List[float] = field(default_factory=list)


@dataclass
class Picker:
    """Picker attributes and the task state used by the operational simulation."""

    id: int
    speed: float
    capacity: float
    current_location: int = 0
    status: str = "idle"
    current_task: Optional[Dict[str, Any]] = None
    performance_history: List[float] = field(default_factory=list)


@dataclass
class Transaction:
    """One simulated SKU-picking task, including its timing fields."""

    timestamp: datetime
    sku_id: int
    quantity: int
    picker_id: int
    from_location: int
    to_location: int
    picking_time: float
    travel_time: float


@dataclass
class SystemKPI:
    """System indicators recorded at a simulation time step."""

    timestamp: datetime
    total_orders: int
    avg_picking_time: float
    picker_utilization: float
    space_utilization: float
    throughput: float
    bottleneck_locations: List[int]
    demand_forecast_error: float

## 1. Build the warehouse simulation

The data classes above separate the properties of SKUs, locations, pickers, transactions, and KPI records. The `WarehouseDigitalTwin` class brings these objects together. It initializes a feasible-by-capacity starting assignment using a velocity-based greedy rule, generates synthetic orders, advances picker tasks in 30-second time steps, and records operational indicators.

A discrete time step is a short interval in which the simulation generates orders, assigns available pickers, updates tasks and sensor readings, and records KPIs. This makes the operational sequence explicit, but it is not a detailed physical simulation. In particular, accessibility multiplied by 10 is only a travel-distance proxy, and the model does not represent aisle routes or picker congestion.

In [ ]:
class WarehouseDigitalTwin:
    """Synthetic warehouse model for slotting and operational what-if analysis."""

    def __init__(self, num_skus=500, num_locations=50, num_pickers=20):
        print(
            f"Initializing warehouse model: {num_skus} SKUs, "
            f"{num_locations} locations, {num_pickers} pickers"
        )
        self.skus = self._create_skus(num_skus)
        self.locations = self._create_locations(num_locations)
        self.pickers = self._create_pickers(num_pickers)

        self.transactions = deque(maxlen=10000)
        self.kpi_history = deque(maxlen=1000)
        self.sensor_data = defaultdict(dict)
        self.current_time = datetime.now()
        self.simulation_speed = 60
        self.total_orders_processed = 0
        self.demand_forecaster = None
        self.bottleneck_predictor = None
        self.current_slotting_plan = {}
        self.pending_relocations = []
        self.optimization_schedule = []
        self._initialize_slotting()
        print(
            f"Warehouse model initialized with {len(self.skus)} SKUs and "
            f"{len(self.locations)} locations"
        )

    def _create_skus(self, num_skus: int) -> List[DigitalSKU]:
        """Generate synthetic SKU characteristics and 24 demand observations."""
        categories = [
            "Electronics", "Clothing", "Food", "Books",
            "Toys", "Sports", "Home", "Beauty"
        ]
        skus = []

        for sku_id in range(num_skus):
            category = np.random.choice(categories)
            if category == "Food":
                velocity = np.random.lognormal(4.5, 0.8)
            elif category == "Electronics":
                velocity = np.random.lognormal(3.5, 1.2)
            else:
                velocity = np.random.lognormal(3.0, 1.0)

            velocity = max(1, velocity)
            skus.append(
                DigitalSKU(
                    id=sku_id + 1,
                    velocity=velocity,
                    space_req=np.random.uniform(0.1, 4.0),
                    weight=np.random.uniform(0.1, 50),
                    category=category,
                    price=np.random.uniform(5, 500),
                    demand_history=[
                        max(1, velocity + np.random.normal(0, velocity * 0.2))
                        for _ in range(24)
                    ],
                )
            )
        return skus

    def _create_locations(self, num_locations: int) -> List[DigitalLocation]:
        """Generate synthetic locations with zone-dependent capacities and access."""
        zones = ["Fast-Pick", "Reserve", "Cold-Storage", "High-Security", "Bulk"]
        locations = []

        for location_id in range(num_locations):
            zone = np.random.choice(zones, p=[0.3, 0.4, 0.1, 0.1, 0.1])
            if zone == "Fast-Pick":
                capacity = np.random.uniform(8, 20)
                accessibility = np.random.uniform(1, 3)
            elif zone == "Reserve":
                capacity = np.random.uniform(15, 40)
                accessibility = np.random.uniform(4, 8)
            else:
                capacity = np.random.uniform(20, 50)
                accessibility = np.random.uniform(5, 10)

            locations.append(
                DigitalLocation(
                    id=location_id + 1,
                    capacity=capacity,
                    weight_limit=capacity * 25,
                    accessibility=accessibility,
                    zone=zone,
                    sensor_data={
                        "temperature": np.random.uniform(15, 25),
                        "humidity": np.random.uniform(30, 70),
                        "last_activity": datetime.now(),
                    },
                )
            )
        return locations

    def _create_pickers(self, num_pickers: int) -> List[Picker]:
        """Generate pickers with synthetic speeds and performance histories."""
        return [
            Picker(
                id=picker_id + 1,
                speed=np.random.uniform(0.8, 1.5),
                capacity=np.random.uniform(20, 50),
                performance_history=[np.random.normal(1.0, 0.1) for _ in range(100)],
            )
            for picker_id in range(num_pickers)
        ]

    def _initialize_slotting(self):
        """Assign high-velocity SKUs first, subject to space and weight limits."""
        print("Initializing the velocity-based slotting assignment...")
        sorted_skus = sorted(self.skus, key=lambda sku: sku.velocity, reverse=True)
        sorted_locations = sorted(self.locations, key=lambda loc: loc.accessibility)

        for sku in sorted_skus:
            for location in sorted_locations:
                space_fits = (
                    location.current_space_used + sku.space_req
                    <= location.capacity
                )
                weight_fits = (
                    location.current_weight_used + sku.weight
                    <= location.weight_limit
                )
                if space_fits and weight_fits:
                    sku.location_id = location.id
                    location.assigned_skus.append(sku.id)
                    location.current_space_used += sku.space_req
                    location.current_weight_used += sku.weight
                    self.current_slotting_plan[sku.id] = location.id
                    break

        print(
            f"Slotting initialized: {len(self.current_slotting_plan)}/"
            f"{len(self.skus)} SKUs assigned"
        )

    def update_demand_patterns(self):
        """Apply synthetic trend, daily seasonality, and noise to SKU velocity."""
        for sku in self.skus:
            trend = np.random.normal(0, 0.01)
            seasonal_factor = 1 + 0.3 * np.sin(
                2 * np.pi * self.current_time.hour / 24
            )
            noise = np.random.normal(0, 0.1)
            new_velocity = max(
                1, sku.velocity * (1 + trend) * seasonal_factor * (1 + noise)
            )
            sku.velocity = new_velocity
            sku.demand_history.append(new_velocity)
            if len(sku.demand_history) > 48:
                sku.demand_history.pop(0)

    def simulate_picking_operations(self, duration_minutes: int = 60):
        """Advance order generation and picker tasks in 30-second time steps."""
        print(f"Simulating {duration_minutes} minutes of picking operations...")
        end_time = self.current_time + timedelta(minutes=duration_minutes)

        while self.current_time < end_time:
            self._generate_orders()
            self._process_picker_tasks()
            self._update_sensor_data()
            self._calculate_kpis()
            self.current_time += timedelta(seconds=30)

        print(
            f"Simulation completed. {len(self.transactions)} transactions "
            "remain in the transaction buffer."
        )

    def _generate_orders(self):
        """Generate synthetic single-SKU picking tasks in proportion to velocity."""
        hour_factor = 1.0 + 0.5 * np.sin(
            2 * np.pi * (self.current_time.hour - 6) / 24
        )
        base_order_rate = len(self.skus) * 0.001
        num_orders = np.random.poisson(base_order_rate * hour_factor)
        sku_weights = np.array([sku.velocity for sku in self.skus], dtype=float)

        for _ in range(num_orders):
            if not self.skus or sku_weights.sum() <= 0:
                return
            sku = np.random.choice(self.skus, p=sku_weights / sku_weights.sum())
            if sku.location_id is not None:
                self.transactions.append(
                    Transaction(
                        timestamp=self.current_time,
                        sku_id=sku.id,
                        quantity=int(np.random.randint(1, 5)),
                        picker_id=0,
                        from_location=sku.location_id,
                        to_location=0,
                        picking_time=0.0,
                        travel_time=0.0,
                    )
                )

    def _process_picker_tasks(self):
        """Assign waiting tasks and advance active tasks toward completion."""
        idle_pickers = [picker for picker in self.pickers if picker.status == "idle"]
        pending_transactions = [
            transaction
            for transaction in self.transactions
            if transaction.picker_id == 0
        ]

        for picker, transaction in zip(idle_pickers, pending_transactions):
            transaction.picker_id = picker.id
            picker.current_task = {
                "transaction": transaction,
                "start_time": self.current_time,
                "status": "traveling",
            }
            picker.status = "picking"

        location_by_id = {location.id: location for location in self.locations}
        for picker in self.pickers:
            if picker.status != "picking" or picker.current_task is None:
                continue

            task = picker.current_task
            transaction = task["transaction"]
            if task["status"] == "traveling":
                location = location_by_id[transaction.from_location]
                travel_distance = location.accessibility * 10
                travel_time = travel_distance / picker.speed
                elapsed = (self.current_time - task["start_time"]).total_seconds()
                if elapsed >= travel_time:
                    task["status"] = "picking"
                    transaction.travel_time = travel_time

            elif task["status"] == "picking":
                picking_time = transaction.quantity * 2
                elapsed = (self.current_time - task["start_time"]).total_seconds()
                if elapsed >= transaction.travel_time + picking_time:
                    transaction.picking_time = picking_time
                    self.total_orders_processed += 1
                    picker.status = "idle"
                    picker.current_task = None
                    picker.performance_history.append(1.0)

    def _update_sensor_data(self):
        """Update synthetic environmental readings and recent location activity."""
        for location in self.locations:
            recent_activity = sum(
                transaction.from_location == location.id
                and (self.current_time - transaction.timestamp).total_seconds() < 3600
                for transaction in self.transactions
            )
            location.sensor_data.update(
                {
                    "temperature": (
                        location.sensor_data["temperature"]
                        + np.random.normal(0, 0.5)
                    ),
                    "humidity": max(
                        0,
                        min(
                            100,
                            location.sensor_data["humidity"]
                            + np.random.normal(0, 2),
                        ),
                    ),
                    "last_activity": self.current_time,
                    "activity_count": recent_activity,
                }
            )
            utilization = (
                location.current_space_used / location.capacity * 100
                if location.capacity > 0
                else 0
            )
            location.utilization_history.append(utilization)
            if len(location.utilization_history) > 24:
                location.utilization_history.pop(0)

    def _calculate_kpis(self):
        """Record recent completed-task metrics and a simple forecast error."""
        recent_completed = [
            transaction
            for transaction in self.transactions
            if transaction.picking_time > 0
            and (self.current_time - transaction.timestamp).total_seconds() < 3600
        ]
        avg_picking_time = (
            float(np.mean([item.picking_time for item in recent_completed]))
            if recent_completed
            else 0.0
        )
        active_pickers = sum(picker.status == "picking" for picker in self.pickers)
        picker_utilization = (
            active_pickers / len(self.pickers) * 100 if self.pickers else 0.0
        )
        total_space = sum(location.capacity for location in self.locations)
        used_space = sum(location.current_space_used for location in self.locations)
        space_utilization = used_space / total_space * 100 if total_space else 0.0
        throughput = float(len(recent_completed))

        location_activities = {
            location.id: sum(
                transaction.from_location == location.id
                for transaction in recent_completed
            )
            for location in self.locations
        }
        positive_activities = [
            activity for activity in location_activities.values() if activity > 0
        ]
        if positive_activities:
            threshold = float(np.percentile(positive_activities, 80))
            bottlenecks = [
                location_id
                for location_id, activity in location_activities.items()
                if activity > 0 and activity >= threshold
            ]
        else:
            bottlenecks = []

        forecast_errors = []
        for sku in self.skus[:50]:
            if len(sku.demand_history) >= 24:
                actual = sku.demand_history[-1]
                predicted = float(np.mean(sku.demand_history[-24:-1]))
                if predicted > 0:
                    forecast_errors.append(abs(actual - predicted) / predicted)

        self.kpi_history.append(
            SystemKPI(
                timestamp=self.current_time,
                total_orders=self.total_orders_processed,
                avg_picking_time=avg_picking_time,
                picker_utilization=picker_utilization,
                space_utilization=space_utilization,
                throughput=throughput,
                bottleneck_locations=bottlenecks,
                demand_forecast_error=(
                    float(np.mean(forecast_errors)) if forecast_errors else 0.0
                ),
            )
        )

## 2. Forecast demand, inspect bottlenecks, and define slotting scenarios

The next class extension adds the analytical and scenario methods. The demand forecast uses the last 12 observations when available. At each forecast step it combines a smoothed level, a simple recent trend, and a sinusoidal daily factor. This is a transparent teaching baseline, not a fitted or validated forecasting model.

Bottleneck analysis separates observed high-activity locations from locations whose forecast-based load exceeds a specified threshold. The latter uses average forecast demand as a simplified load proxy; it does not model inventory replenishment or physical congestion. Scenario generation uses the greedy rules described above and checks space and weight feasibility.

In [ ]:
class WarehouseDigitalTwin(WarehouseDigitalTwin):
    """Add forecasts, bottleneck analysis, and greedy slotting scenarios."""

    def forecast_demand_trends(self, hours_ahead: int = 24) -> Dict[int, List[float]]:
        """Forecast each SKU with smoothing, recent trend, and daily seasonality."""
        print(f"Forecasting demand trends for the next {hours_ahead} hours...")
        forecasts = {}
        alpha = 0.3

        for sku in self.skus:
            if len(sku.demand_history) < 12:
                continue
            history = sku.demand_history[-12:]
            forecast = []
            current_value = history[-1]
            recent_trend = float(np.mean(np.diff(history[-4:])))

            for hour in range(hours_ahead):
                seasonal_factor = 1 + 0.2 * np.sin(
                    2 * np.pi * (self.current_time.hour + hour + 1) / 24
                )
                if hour == 0:
                    next_value = alpha * current_value + (1 - alpha) * np.mean(history)
                else:
                    next_value = alpha * current_value + (1 - alpha) * forecast[-1]
                forecast_value = max(1, next_value * seasonal_factor + recent_trend)
                forecast.append(float(forecast_value))
                current_value = forecast_value

            forecasts[sku.id] = forecast

        print(f"Generated forecasts for {len(forecasts)} SKUs")
        return forecasts

    def identify_bottlenecks(self) -> Dict[str, Any]:
        """Report observed activity hotspots and forecast-based load estimates."""
        print("Identifying observed and forecast-based bottlenecks...")
        recent_kpi = self.kpi_history[-1] if self.kpi_history else None
        current_bottlenecks = (
            recent_kpi.bottleneck_locations if recent_kpi is not None else []
        )
        demand_forecasts = self.forecast_demand_trends(6)
        sku_by_id = {sku.id: sku for sku in self.skus}
        predicted_loads = defaultdict(float)

        for sku_id, forecast in demand_forecasts.items():
            sku = sku_by_id[sku_id]
            if sku.location_id is not None:
                predicted_loads[sku.location_id] += float(np.mean(forecast))

        location_by_id = {location.id: location for location in self.locations}
        predicted_utilizations = {}
        for location_id, load in predicted_loads.items():
            location = location_by_id[location_id]
            if location.capacity > 0:
                predicted_utilizations[location_id] = load / location.capacity * 100

        high_utilization_threshold = 85
        predicted_bottlenecks = [
            location_id
            for location_id, utilization in predicted_utilizations.items()
            if utilization > high_utilization_threshold
        ]
        recommendations = []

        for location_id in predicted_bottlenecks:
            location = location_by_id[location_id]
            high_velocity_skus = [
                sku_id
                for sku_id in location.assigned_skus
                if sku_id in sku_by_id and sku_by_id[sku_id].velocity > 50
            ]
            if high_velocity_skus:
                utilization = predicted_utilizations[location_id]
                recommendations.append(
                    {
                        "location_id": location_id,
                        "issue": "High forecast-based load estimate",
                        "suggested_action": (
                            f"Review {len(high_velocity_skus)} high-velocity SKUs "
                            "assigned to this location"
                        ),
                        "priority": "High" if utilization > 95 else "Medium",
                    }
                )

        analysis = {
            "current_bottlenecks": current_bottlenecks,
            "predicted_bottlenecks": predicted_bottlenecks,
            "predicted_utilizations": predicted_utilizations,
            "recommendations": recommendations,
        }
        print(
            f"Identified {len(current_bottlenecks)} observed high-activity and "
            f"{len(predicted_bottlenecks)} forecast-based locations"
        )
        return analysis

    def optimize_slotting_scenario(
        self, scenario_name: str, optimization_target: str = "efficiency"
    ) -> Dict[str, Any]:
        """Build and assess one heuristic slotting scenario in a copied state."""
        print(f"Running slotting scenario: {scenario_name}")
        print(f"Scenario target: {optimization_target}")
        simulation_state = self._create_simulation_copy()

        if optimization_target == "efficiency":
            assignments = self._optimize_for_efficiency(simulation_state)
        elif optimization_target == "space_utilization":
            assignments = self._optimize_for_space_utilization(simulation_state)
        elif optimization_target == "balanced":
            assignments = self._optimize_balanced(simulation_state)
        else:
            assignments = simulation_state["current_assignments"]

        performance = self._simulate_scenario_performance(
            simulation_state, assignments
        )
        current_performance = self._get_current_performance_metrics()
        improvements = self._calculate_improvements(
            current_performance, performance
        )
        results = {
            "scenario_name": scenario_name,
            "optimization_target": optimization_target,
            "optimized_assignments": assignments,
            "performance_metrics": performance,
            "current_performance": current_performance,
            "improvements": improvements,
            "implementation_complexity": self._assess_implementation_complexity(
                assignments
            ),
            "estimated_disruption": self._estimate_operational_disruption(
                assignments
            ),
        }
        print(
            "Scenario completed. Model-based composite improvement estimate: "
            f"{improvements['overall_improvement']:.1f}%"
        )
        return results

    def _create_simulation_copy(self) -> Dict[str, Any]:
        """Copy only the attributes required by the slotting heuristics."""
        return {
            "skus": [
                {
                    "id": sku.id,
                    "velocity": sku.velocity,
                    "space_req": sku.space_req,
                    "weight": sku.weight,
                }
                for sku in self.skus
            ],
            "locations": [
                {
                    "id": location.id,
                    "capacity": location.capacity,
                    "weight_limit": location.weight_limit,
                    "accessibility": location.accessibility,
                    "zone": location.zone,
                }
                for location in self.locations
            ],
            "current_assignments": self.current_slotting_plan.copy(),
        }

    def _optimize_for_efficiency(
        self, simulation_state: Dict[str, Any]
    ) -> Dict[int, int]:
        """Prioritize high-velocity SKUs and accessible feasible locations."""
        skus = sorted(
            simulation_state["skus"],
            key=lambda sku: sku["velocity"],
            reverse=True,
        )
        locations = sorted(
            simulation_state["locations"],
            key=lambda location: location["accessibility"],
        )
        assignments = {}
        usage = defaultdict(lambda: {"space": 0.0, "weight": 0.0})

        for sku in skus:
            for location in locations:
                location_id = location["id"]
                if (
                    usage[location_id]["space"] + sku["space_req"]
                    <= location["capacity"]
                    and usage[location_id]["weight"] + sku["weight"]
                    <= location["weight_limit"]
                ):
                    assignments[sku["id"]] = location_id
                    usage[location_id]["space"] += sku["space_req"]
                    usage[location_id]["weight"] += sku["weight"]
                    break
        return assignments

    def _optimize_for_space_utilization(
        self, simulation_state: Dict[str, Any]
    ) -> Dict[int, int]:
        """Place each SKU in a feasible location with the lowest current utilization."""
        skus = simulation_state["skus"]
        locations = simulation_state["locations"]
        assignments = {}
        usage = defaultdict(lambda: {"space": 0.0, "weight": 0.0})
        shuffled_skus = skus.copy()
        np.random.shuffle(shuffled_skus)

        for sku in shuffled_skus:
            best_location = None
            lowest_utilization = float("inf")
            for location in locations:
                location_id = location["id"]
                current_utilization = (
                    usage[location_id]["space"] / location["capacity"] * 100
                    if location["capacity"] > 0
                    else float("inf")
                )
                feasible = (
                    usage[location_id]["space"] + sku["space_req"]
                    <= location["capacity"]
                    and usage[location_id]["weight"] + sku["weight"]
                    <= location["weight_limit"]
                )
                if feasible and current_utilization < lowest_utilization:
                    lowest_utilization = current_utilization
                    best_location = location

            if best_location is not None:
                location_id = best_location["id"]
                assignments[sku["id"]] = location_id
                usage[location_id]["space"] += sku["space_req"]
                usage[location_id]["weight"] += sku["weight"]
        return assignments

    def _optimize_balanced(
        self, simulation_state: Dict[str, Any]
    ) -> Dict[int, int]:
        """Balance accessibility priority with a penalty for already-used capacity."""
        skus = simulation_state["skus"]
        locations = simulation_state["locations"]
        min_accessibility = min(
            location["accessibility"] for location in locations
        )
        max_accessibility = max(
            location["accessibility"] for location in locations
        )
        location_scores = {}
        for location in locations:
            if max_accessibility > min_accessibility:
                normalized = (
                    location["accessibility"] - min_accessibility
                ) / (max_accessibility - min_accessibility)
            else:
                normalized = 0.0
            location_scores[location["id"]] = 1 - normalized

        priorities = [
            (sku, sku["velocity"] / sku["space_req"])
            for sku in skus
        ]
        priorities.sort(key=lambda item: item[1], reverse=True)
        assignments = {}
        usage = defaultdict(lambda: {"space": 0.0, "weight": 0.0})

        for sku, _priority in priorities:
            best_location = None
            best_score = float("-inf")
            for location in locations:
                location_id = location["id"]
                feasible = (
                    usage[location_id]["space"] + sku["space_req"]
                    <= location["capacity"]
                    and usage[location_id]["weight"] + sku["weight"]
                    <= location["weight_limit"]
                )
                if not feasible:
                    continue
                current_utilization = (
                    usage[location_id]["space"] / location["capacity"]
                    if location["capacity"] > 0
                    else 0.0
                )
                score = location_scores[location_id] - 0.4 * current_utilization
                if score > best_score:
                    best_score = score
                    best_location = location

            if best_location is not None:
                location_id = best_location["id"]
                assignments[sku["id"]] = location_id
                usage[location_id]["space"] += sku["space_req"]
                usage[location_id]["weight"] += sku["weight"]
        return assignments

## 3. Establish a simulated baseline

The following cell uses the example size specified for this notebook. Order generation is random but reproducible under the fixed seeds, and the simulation advances in half-minute increments. The results depend on the synthetic assumptions and on the environment's random-number and plotting-library behavior; no precomputed output is supplied.

In [ ]:
digital_twin = WarehouseDigitalTwin(
    num_skus=200,
    num_locations=30,
    num_pickers=10,
)

print("=== BASELINE OPERATIONS ===")
digital_twin.simulate_picking_operations(duration_minutes=120)

## 4. Inspect current performance

A key performance indicator (KPI) is a numerical summary of an aspect of system behavior. Here, picking time is the mean picking duration of recently completed tasks; picker utilization is the share of pickers currently marked active; space utilization is occupied modeled space divided by total modeled capacity; and throughput is the count of completed tasks in the trailing hour. The location-activity chart counts transactions in the buffer, while the KPI bottleneck list uses completed recent tasks. These measures describe this simulation's definitions, not universal warehouse KPI standards.

In [ ]:
def analyze_current_performance(digital_twin: WarehouseDigitalTwin):
    """Print and plot the latest KPI record and selected warehouse summaries."""
    print("\n=== CURRENT SYSTEM PERFORMANCE ===")
    if not digital_twin.kpi_history:
        print("No performance records are available.")
        return None

    latest_kpi = digital_twin.kpi_history[-1]
    print(f"Performance timestamp: {latest_kpi.timestamp.strftime('%H:%M:%S')}")
    print(f"Total completed tasks: {latest_kpi.total_orders}")
    print(f"Mean picking time: {latest_kpi.avg_picking_time:.2f} seconds")
    print(f"Active picker share: {latest_kpi.picker_utilization:.1f}%")
    print(f"Modeled space utilization: {latest_kpi.space_utilization:.1f}%")
    print(f"Completed tasks in trailing hour: {latest_kpi.throughput:.1f}")
    print(f"Observed high-activity locations: {latest_kpi.bottleneck_locations}")
    print(f"Simple demand forecast error: {latest_kpi.demand_forecast_error:.3f}")

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    timestamps = [kpi.timestamp for kpi in digital_twin.kpi_history]
    axes[0, 0].plot(
        timestamps,
        [kpi.avg_picking_time for kpi in digital_twin.kpi_history],
        color="blue",
        linewidth=2,
    )
    axes[0, 0].set_title("Mean Picking Time")
    axes[0, 0].set_ylabel("Seconds")
    axes[0, 0].set_xlabel("Simulation time")
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(
        timestamps,
        [kpi.picker_utilization for kpi in digital_twin.kpi_history],
        color="green",
        linewidth=2,
    )
    axes[0, 1].set_title("Active Picker Share")
    axes[0, 1].set_ylabel("Percent")
    axes[0, 1].set_xlabel("Simulation time")
    axes[0, 1].grid(True, alpha=0.3)

    axes[0, 2].plot(
        timestamps,
        [kpi.space_utilization for kpi in digital_twin.kpi_history],
        color="red",
        linewidth=2,
    )
    axes[0, 2].set_title("Modeled Space Utilization")
    axes[0, 2].set_ylabel("Percent")
    axes[0, 2].set_xlabel("Simulation time")
    axes[0, 2].grid(True, alpha=0.3)

    axes[1, 0].plot(
        timestamps,
        [kpi.throughput for kpi in digital_twin.kpi_history],
        color="purple",
        linewidth=2,
    )
    axes[1, 0].set_title("Completed Tasks in Trailing Hour")
    axes[1, 0].set_ylabel("Tasks")
    axes[1, 0].set_xlabel("Simulation time")
    axes[1, 0].grid(True, alpha=0.3)

    location_activities = defaultdict(int)
    for transaction in digital_twin.transactions:
        location_activities[transaction.from_location] += 1
    if location_activities:
        location_ids = sorted(location_activities)
        activities = [location_activities[item] for item in location_ids]
        axes[1, 1].bar(location_ids, activities, color="orange", alpha=0.7)
        axes[1, 1].set_title("Transaction Count by Location")
        axes[1, 1].set_xlabel("Location ID")
        axes[1, 1].set_ylabel("Transactions in buffer")
        axes[1, 1].grid(True, alpha=0.3)
    else:
        axes[1, 1].set_title("Transaction Count by Location")
        axes[1, 1].text(0.5, 0.5, "No transactions", ha="center", va="center")

    zone_utilization = defaultdict(list)
    for location in digital_twin.locations:
        if location.capacity > 0:
            utilization = location.current_space_used / location.capacity * 100
            zone_utilization[location.zone].append(utilization)
    zone_averages = {
        zone: float(np.mean(values))
        for zone, values in zone_utilization.items()
    }
    axes[1, 2].bar(
        list(zone_averages),
        list(zone_averages.values()),
        color="lightgreen",
        alpha=0.7,
    )
    axes[1, 2].set_title("Mean Utilization by Zone")
    axes[1, 2].set_ylabel("Percent")
    axes[1, 2].set_xlabel("Zone")
    axes[1, 2].tick_params(axis="x", rotation=45)
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return latest_kpi


current_performance = analyze_current_performance(digital_twin)

## 5. Forecast demand and inspect bottleneck estimates

A forecast is an estimate of future values based on past observations and an explicit rule. The forecast function returns one sequence per SKU. The visualization shows a sample of SKU histories and their forecast sequences. Its shaded band is a descriptive range formed from 20% of the historical standard deviation; it is not a statistically calibrated prediction interval.

The bottleneck analysis reports current high-activity locations from the latest KPI record and forecast-based locations whose simplified load estimate exceeds 85% of modeled capacity. These are two different signals and should not be treated as proof of congestion.

In [ ]:
print("\n=== DEMAND FORECAST AND BOTTLENECK ANALYSIS ===")
demand_forecasts = digital_twin.forecast_demand_trends(hours_ahead=12)
bottleneck_analysis = digital_twin.identify_bottlenecks()

print("\nBottleneck analysis:")
print(f"Observed high-activity locations: {bottleneck_analysis['current_bottlenecks']}")
print(f"Forecast-based locations: {bottleneck_analysis['predicted_bottlenecks']}")
print("Recommendations:")
for index, recommendation in enumerate(
    bottleneck_analysis["recommendations"][:5], start=1
):
    print(
        f"{index}. Location {recommendation['location_id']}: "
        f"{recommendation['suggested_action']} "
        f"(Priority: {recommendation['priority']})"
    )


def visualize_demand_forecasts(
    digital_twin: WarehouseDigitalTwin,
    demand_forecasts: Dict[int, List[float]],
):
    """Plot historical demand and forecasts for a reproducible sample of SKUs."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    sample_size = min(4, len(digital_twin.skus))
    sample_skus = random.sample(digital_twin.skus, sample_size)

    for index, sku in enumerate(sample_skus):
        axis = axes[index // 2, index % 2]
        if sku.id not in demand_forecasts:
            axis.text(
                0.5,
                0.5,
                f"No forecast available for SKU {sku.id}",
                ha="center",
                va="center",
                transform=axis.transAxes,
            )
            axis.set_title(f"SKU {sku.id} Demand")
            continue

        historical_hours = np.arange(len(sku.demand_history))
        forecast_hours = np.arange(
            len(sku.demand_history),
            len(sku.demand_history) + len(demand_forecasts[sku.id]),
        )
        axis.plot(
            historical_hours,
            sku.demand_history,
            color="blue",
            label="Synthetic history",
            linewidth=2,
        )
        axis.plot(
            forecast_hours,
            demand_forecasts[sku.id],
            color="red",
            linestyle="--",
            label="Forecast",
            linewidth=2,
        )
        forecast_spread = float(np.std(sku.demand_history) * 0.2)
        axis.fill_between(
            forecast_hours,
            np.array(demand_forecasts[sku.id]) - forecast_spread,
            np.array(demand_forecasts[sku.id]) + forecast_spread,
            alpha=0.3,
            color="red",
            label="Descriptive range (±20% of historical SD)",
        )
        axis.set_title(f"SKU {sku.id} Demand")
        axis.set_xlabel("Observation / forecast step")
        axis.set_ylabel("Synthetic demand rate")
        axis.legend()
        axis.grid(True, alpha=0.3)

    for index in range(sample_size, 4):
        axes[index // 2, index % 2].set_visible(False)
    plt.tight_layout()
    plt.show()


visualize_demand_forecasts(digital_twin, demand_forecasts)

## 6. Define scenario comparisons

The scenario comparison uses three assignment heuristics: efficiency-focused, space-utilization-focused, and balanced. Each returns an SKU-to-location mapping for the SKUs it can place under the modeled capacity constraints. The comparison function plots the resulting model-based measures and prints a summary. Its implementation and interpretation are completed after the common performance calculations are defined.

In [ ]:
scenarios = [
    ("Efficiency_Focused", "efficiency"),
    ("Space_Balanced", "space_utilization"),
    ("Balanced_Optimization", "balanced"),
]


def compare_scenarios(scenario_results: List[Dict[str, Any]]):
    """Plot model-based scenario estimates and print their calculated summaries."""
    if not scenario_results:
        print("No scenario results are available to compare.")
        return

    print("\n=== SCENARIO COMPARISON ===")
    names = [result["scenario_name"] for result in scenario_results]
    improvements = [
        result["improvements"]["overall_improvement"]
        for result in scenario_results
    ]
    complexities = [
        result["implementation_complexity"] for result in scenario_results
    ]
    disruptions = [
        result["estimated_disruption"] for result in scenario_results
    ]

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    bars = axes[0, 0].bar(names, improvements, color="skyblue", alpha=0.7)
    axes[0, 0].set_title("Composite Model-Based Improvement Estimate")
    axes[0, 0].set_ylabel("Weighted score (%)")
    axes[0, 0].set_xlabel("Scenario")
    axes[0, 0].tick_params(axis="x", rotation=15)
    axes[0, 0].grid(True, alpha=0.3)
    for bar, value in zip(bars, improvements):
        offset = 0.01 * max(1.0, abs(value))
        axes[0, 0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            f"{value:.1f}%",
            ha="center",
            va="bottom",
            fontweight="bold",
        )

    complexity_levels = ["Low", "Medium", "High"]
    complexity_counts = {
        level: complexities.count(level) for level in complexity_levels
    }
    axes[0, 1].bar(
        complexity_levels,
        [complexity_counts[level] for level in complexity_levels],
        color="lightgreen",
        alpha=0.7,
    )
    axes[0, 1].set_title("Scenario Implementation-Complexity Categories")
    axes[0, 1].set_xlabel("Heuristic category")
    axes[0, 1].set_ylabel("Number of scenarios")
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].bar(names, disruptions, color="orange", alpha=0.7)
    axes[1, 0].set_title("Estimated Relocation Disruption")
    axes[1, 0].set_xlabel("Scenario")
    axes[1, 0].set_ylabel("Heuristic score (0–10)")
    axes[1, 0].tick_params(axis="x", rotation=15)
    axes[1, 0].grid(True, alpha=0.3)

    metrics = [
        ("distance_improvement", "Distance-proxy improvement"),
        ("space_utilization_improvement", "Space-utilization change"),
        ("throughput_improvement", "Estimated-throughput change"),
        ("picking_time_improvement", "Estimated-picking-time change"),
    ]
    positions = np.arange(len(names))
    width = 0.2
    for metric_index, (metric_key, label) in enumerate(metrics):
        values = [
            result["improvements"].get(metric_key, 0.0)
            for result in scenario_results
        ]
        axes[1, 1].bar(
            positions + metric_index * width,
            values,
            width,
            label=label,
        )
    axes[1, 1].set_title("Component Measures")
    axes[1, 1].set_xlabel("Scenario")
    axes[1, 1].set_ylabel("Change (%) or percentage points")
    axes[1, 1].set_xticks(positions + width * 1.5)
    axes[1, 1].set_xticklabels(names, rotation=15)
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\nScenario summary:")
    for result in scenario_results:
        print(f"\n{result['scenario_name']}:")
        print(
            "  Composite model-based estimate: "
            f"{result['improvements']['overall_improvement']:.1f}%"
        )
        print(
            "  Estimated implementation complexity: "
            f"{result['implementation_complexity']}"
        )
        print(
            "  Estimated relocation disruption score: "
            f"{result['estimated_disruption']:.1f}/10"
        )
        for key, label in metrics:
            if key in result["improvements"]:
                print(f"  {label}: {result['improvements'][key]:.1f}")

## 7. Calculate scenario performance and run the comparison

For each candidate assignment, the performance routine sums demand-weighted accessibility-distance proxies, computes space utilization across all modeled locations, and estimates throughput from a distance proxy. In the code, `avg_distance` is the total demand-weighted distance divided by the number of assigned SKUs; it is therefore a mean contribution per assigned SKU, not a conventional distance averaged by total demand. The resulting throughput and picking-time values are simplified model proxies, not outputs from a second picker-operation simulation.

The composite improvement is a weighted average of available component measures. Distance and estimated picking time are treated as lower-is-better; estimated throughput is higher-is-better; and space utilization is reported as a nonnegative percentage-point increase only when the candidate utilization is at most 95%. Because these measures have different meanings and scales, the composite is a heuristic summary rather than a calibrated economic objective. The disruption score is also a heuristic based on relocation and zone changes.

The forecast-based location score in the bottleneck analysis divides average forecast demand by modeled space capacity. It is a screening proxy, not physical space utilization: forecast demand and capacity do not share a physical unit in this synthetic model. Its threshold is consequently a heuristic score threshold, not a measured congestion or occupancy limit.

In [ ]:
class WarehouseDigitalTwin(WarehouseDigitalTwin):
    """Add common scenario metrics and relocation-impact estimates."""

    def _simulate_scenario_performance(
        self,
        simulation_state: Dict[str, Any],
        assignments: Dict[int, int],
    ) -> Dict[str, float]:
        """Evaluate an assignment using distance and capacity-based proxies."""
        sku_by_id = {sku["id"]: sku for sku in simulation_state["skus"]}
        location_by_id = {
            location["id"]: location for location in simulation_state["locations"]
        }
        total_weighted_distance = 0.0
        location_usage = defaultdict(float)

        for sku_id, location_id in assignments.items():
            sku = sku_by_id[sku_id]
            location = location_by_id[location_id]
            distance = location["accessibility"] * 10
            total_weighted_distance += sku["velocity"] * distance
            location_usage[location_id] += sku["space_req"]

        total_space_used = sum(location_usage.values())
        total_capacity = sum(
            location["capacity"] for location in simulation_state["locations"]
        )
        space_utilization = (
            total_space_used / total_capacity * 100 if total_capacity else 0.0
        )
        avg_distance = (
            total_weighted_distance / len(assignments) if assignments else 0.0
        )
        estimated_throughput = (
            3600 / (avg_distance + 30) if avg_distance > 0 else 100.0
        )
        estimated_picking_time = avg_distance * 0.1

        return {
            "total_distance": total_weighted_distance,
            "space_utilization": space_utilization,
            "throughput": estimated_throughput,
            "avg_distance": avg_distance,
            "avg_picking_time": estimated_picking_time,
            "assigned_skus": float(len(assignments)),
        }

    def _get_current_performance_metrics(self) -> Dict[str, float]:
        """Evaluate the current assignment with the same proxies as each scenario."""
        simulation_state = self._create_simulation_copy()
        return self._simulate_scenario_performance(
            simulation_state,
            simulation_state["current_assignments"],
        )

    def _calculate_improvements(
        self,
        current: Dict[str, float],
        optimized: Dict[str, float],
    ) -> Dict[str, float]:
        """Calculate directional component changes and their heuristic composite."""
        improvements = {}
        if current["total_distance"] > 0:
            improvements["distance_improvement"] = (
                (current["total_distance"] - optimized["total_distance"])
                / current["total_distance"]
                * 100
            )

        if optimized["space_utilization"] <= 95:
            improvements["space_utilization_improvement"] = max(
                0.0,
                optimized["space_utilization"] - current["space_utilization"],
            )

        if current["throughput"] > 0:
            improvements["throughput_improvement"] = (
                (optimized["throughput"] - current["throughput"])
                / current["throughput"]
                * 100
            )

        if current["avg_picking_time"] > 0:
            improvements["picking_time_improvement"] = (
                (current["avg_picking_time"] - optimized["avg_picking_time"])
                / current["avg_picking_time"]
                * 100
            )

        weights = {
            "distance_improvement": 0.3,
            "space_utilization_improvement": 0.2,
            "throughput_improvement": 0.3,
            "picking_time_improvement": 0.2,
        }
        weighted_total = 0.0
        total_weight = 0.0
        for metric, weight in weights.items():
            if metric in improvements:
                weighted_total += improvements[metric] * weight
                total_weight += weight
        improvements["overall_improvement"] = (
            weighted_total / total_weight if total_weight > 0 else 0.0
        )
        return improvements

    def _assess_implementation_complexity(
        self, assignments: Dict[int, int]
    ) -> str:
        """Classify relocation effort by the share of assigned SKUs that move."""
        changes = sum(
            sku_id in self.current_slotting_plan
            and self.current_slotting_plan[sku_id] != new_location_id
            for sku_id, new_location_id in assignments.items()
        )
        change_percentage = changes / len(assignments) * 100 if assignments else 0
        if change_percentage < 10:
            return "Low"
        if change_percentage < 30:
            return "Medium"
        return "High"

    def _estimate_operational_disruption(
        self, assignments: Dict[int, int]
    ) -> float:
        """Estimate a 0–10 relocation score from assignment and zone changes."""
        if not assignments:
            return 0.0

        sku_by_id = {sku.id: sku for sku in self.skus}
        location_by_id = {location.id: location for location in self.locations}
        changes = 0
        zone_changes = 0
        for sku_id, new_location_id in assignments.items():
            old_location_id = self.current_slotting_plan.get(sku_id)
            if old_location_id is None:
                continue
            if old_location_id != new_location_id:
                changes += 1
                old_zone = location_by_id[old_location_id].zone
                new_zone = location_by_id[new_location_id].zone
                if old_zone != new_zone:
                    zone_changes += 1

        assignment_count = len(assignments)
        change_factor = changes / assignment_count * 5
        zone_factor = zone_changes / assignment_count * 3
        return min(10.0, change_factor + zone_factor)


# The baseline instance was created before this class extension. Rebinding its
# Python class exposes the added methods while preserving its simulated state.
digital_twin.__class__ = WarehouseDigitalTwin

print("\n=== RUNNING SLOTTING SCENARIOS ===")
scenario_results = []
for scenario_name, target in scenarios:
    print(f"\n--- {scenario_name} ---")
    scenario_results.append(
        digital_twin.optimize_slotting_scenario(scenario_name, target)
    )

compare_scenarios(scenario_results)

## 8. Results interpretation and limitations

When the notebook is run, inspect the printed KPI record, forecast plots, bottleneck lists, scenario component measures, and assignment counts. Interpret each result in relation to its definition:

- The simulated KPI history describes the synthetic baseline operations, not the candidate scenario assignments.
- Forecasts are generated by a simple smoothing, trend, and seasonal rule. The shaded chart range is descriptive and is not a confidence interval with a stated coverage probability.
- Observed high-activity locations are identified from completed simulated tasks in the trailing hour. Forecast-based locations are identified by a separate demand-to-capacity proxy.
- Scenario distance, throughput, and picking-time values are model-based estimates. Candidate assignments are not passed through a second picker-operation simulation.
- The space-utilization improvement is a percentage-point change subject to the stated 95% cutoff; it is not a relative percentage change.
- A composite improvement combines differently scaled measures using the specified weights. It is useful only as a compact heuristic comparison under these modeling choices, not as a guaranteed operational gain.
- Greedy assignment can leave SKUs unassigned when remaining location capacity is insufficient. Compare assigned SKU counts as well as performance measures before drawing conclusions.

The described MegaFulfill figures are case-description claims and are separate from these synthetic outputs. No result in this notebook establishes real-world forecast accuracy, a realized reduction in picking time, or a particular return on investment.

### Modeling extensions

A more operationally representative study would require validated warehouse geometry, observed order and travel data, appropriate SKU-location eligibility rules, inventory and replenishment behavior, and comparisons between simulation predictions and measured outcomes. Additional constraints or data can be incorporated within the same digital-twin and scenario-analysis workflow; the synthetic model here is a starting point for understanding that workflow, not a substitute for validation.